In [1]:
import pandas as pd
import os
import sys
import tomllib


sys.path.append(os.path.abspath("../.secrets"))
from db_config import get_connection

print ("Se conecto a la db")

Se conecto a la db


In [2]:
with open("../.secrets/secrets.toml", "rb") as f:
 config = tomllib.load(f)

ruta_archivo = config["paths"]["bronze_path"]

print ("Ruta del archivo: ", ruta_archivo)

Ruta del archivo:  assets/docs


In [3]:
conn = get_connection()
cursor = conn.cursor()

print ("Se conecto a la db y se obtuvo el cursor")

Se conecto a la db y se obtuvo el cursor


In [14]:
archivos = [f for f in os.listdir(f"../{ruta_archivo}") if f.endswith(".csv")]

print("archivos disponibles:")
for i, archivo in enumerate(archivos):
 print(f"{i} - {archivo}")

indice = int(input("Seleccione el índice del archivo a cargar: "))

if indice < 0 or indice >= len(archivos):
 print("Índice inválido.")
 exit()

archivo_seleccionado = archivos[indice]
print(f"Archivo seleccionado: {archivo_seleccionado}")
print(f'{archivo}')

archivos disponibles:
0 - clientes.csv
1 - productos.csv
2 - ventas.csv
Archivo seleccionado: clientes.csv
ventas.csv


In [ ]:
from sqlalchemy import create_engine, inspect, text
import pandas as pd

# 1. Configuración del Engine
user = config["database"]["user"]
password = config["database"]["password"]
host = config["database"]["host"]
port = config["database"]["port"]
dbname = config["database"]["dbname"]

engine = create_engine(f'postgresql://{user}:{password}@{host}:{port}/{dbname}')

# 2. Mapeo simplificado
mapeo_config = {
    "ventas.csv": "ventas_raw",
    "productos.csv": "productos_raw",
    "clientes.csv": "clientes_raw"
}

if archivo_seleccionado in mapeo_config:
    tabla = mapeo_config[archivo_seleccionado]
    df = pd.read_csv(f"../{ruta_archivo}/{archivo_seleccionado}")

    # 3. Validación de columnas 
    inspector = inspect(engine)
    columnas_db = [c['name'] for c in inspector.get_columns(tabla, schema='bronze')]
    
    if 'fuente_archivo' in columnas_db:
        df['fuente_archivo'] = archivo_seleccionado
    
    # Solo enviamos las columnas que la DB reconoce
    df_final = df[[c for c in df.columns if c in columnas_db]]

    try:
        # 4. Vaciado y Carga 
        with engine.begin() as conn:
            conn.execute(text(f"TRUNCATE TABLE bronze.{tabla} RESTART IDENTITY CASCADE;"))
            
        df_final.to_sql(
            name=tabla, 
            con=engine, 
            schema='bronze', 
            if_exists='append',
            index=False, 
            method='multi'
        )
        print(f"✅ ¡Éxito! {len(df_final)} filas sincronizadas en bronze.{tabla}")
        
    except Exception as e:
        print(f"❌ Error durante la carga: {e}")

✅ ¡Éxito! 100 filas sincronizadas en bronze.clientes_raw
